# Gemma 4 Trust & Safety Context Pruning Runner

Use this notebook inside Kaggle for Gemma 4 Trust & Safety research experiments.
It runs the ACPA pipeline, creates benchmark artifacts, and writes JSONL output.

Expected Kaggle setup:

1. Attach the Agentic Eval dataset to this notebook.
2. Add your Gemma/Gemini API key through Kaggle Secrets using label `GEMINI_API_KEY`.
3. Turn Kaggle notebook **Internet** on.
4. Click **Run All**.

The first code cell uses the direct GitHub clone/bootstrap path: it clones or updates the `cursor/kaggle-dataset-fallback-34a1` branch in `/kaggle/working/context-pruning`, adds `/kaggle/working/context-pruning/src` to `sys.path`, and verifies that `acpa_gemma` is importable.

Do not paste API keys into public notebook cells or commit them to Git.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

# Direct GitHub bootstrap for Kaggle "Run All".
# Requires Kaggle notebook Internet = On.
REPO_URL = 'https://github.com/joyjeni/context-pruning.git'
REPO_BRANCH = 'cursor/kaggle-dataset-fallback-34a1'
REPO_ROOT = Path('/kaggle/working/context-pruning') if Path('/kaggle/working').exists() else Path.cwd() / 'context-pruning'
SRC_ROOT = REPO_ROOT / 'src'

try:
    if (REPO_ROOT / '.git').exists():
        print('Updating repository branch:', REPO_BRANCH)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', REPO_BRANCH, '--depth', '1'], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
    elif not (SRC_ROOT / 'acpa_gemma').exists():
        if REPO_ROOT.exists():
            print('Removing stale repo directory:', REPO_ROOT)
            shutil.rmtree(REPO_ROOT)
        print('Cloning repository branch:', REPO_BRANCH)
        subprocess.run(
            [
                'git',
                'clone',
                '--depth',
                '1',
                '--branch',
                REPO_BRANCH,
                REPO_URL,
                str(REPO_ROOT),
            ],
            check=True,
        )
except Exception as exc:
    if not (SRC_ROOT / 'acpa_gemma').exists():
        raise RuntimeError(
            'Could not clone/update the repository. Turn Kaggle Notebook Internet ON, '
            'then rerun from the first cell. If Internet is disabled, upload the '
            'repo as a Kaggle dataset and set REPO_ROOT to that dataset path.'
        ) from exc
    print('WARNING: Could not update repository, using existing local copy:', exc)

if not (SRC_ROOT / 'acpa_gemma').exists():
    raise RuntimeError(f'acpa_gemma source package not found under {SRC_ROOT}')

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('REPO_ROOT =', REPO_ROOT)
print('SRC_ROOT exists =', SRC_ROOT.exists())
print('acpa_gemma exists =', (SRC_ROOT / 'acpa_gemma').exists())
print('Kaggle input exists =', Path('/kaggle/input').exists())

import acpa_gemma
print('Imported acpa_gemma from:', acpa_gemma.__file__)


## Dependency install

Run this cell as normal Python. Do **not** type `pip install ...` without a leading `!` in a Kaggle code cell; that causes `SyntaxError`. The cell below uses `python -m pip` through `subprocess`, so it works as valid Python. Internet must be enabled in the notebook settings if packages are missing.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    'google.genai': 'google-genai',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow',
}

missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All required packages are already installed.')


## Locate AgentEval / Agentic Eval dataset

No download is needed in Kaggle after you add the dataset as notebook input. This cell first checks the expected auto-mounted path `/kaggle/input/agent-eval-scenarios`, then prints all attached Kaggle input directories, counts supported files, and selects the directory with the most CSV/JSON/JSONL/NDJSON/Parquet files.

If it chooses the wrong directory, set `MANUAL_DATASET_DIR` in the code cell below to the attached dataset directory before continuing.

If this cell reports `/kaggle/input` with `total_files=0`, no Kaggle input data is attached to the notebook yet. This is not a code error. Open the notebook right sidebar, click **Add data** / **Input**, attach `mukundakatta/agent-eval-scenarios` or upload equivalent AgentEval records, then rerun this cell.

If Kaggle only shows a competition note such as `/kaggle/input/competitions/.../NOTE.md`, the AgentEval records are not attached yet. The dry-run and benchmark cells can still continue with the built-in demo record, but the real Gemma run requires an attached dataset with CSV/JSON/JSONL/NDJSON/Parquet files.


In [ ]:
from pathlib import Path
from acpa_gemma.data import dataset_diagnostics, discover_dataset_files, format_dataset_diagnostics, load_agentic_eval_dataset

# Kaggle auto-mounts this dataset here after you add mukundakatta/agent-eval-scenarios as input.
EXPECTED_DATASET_DIR = Path('/kaggle/input/agent-eval-scenarios')

# Optional manual override. Example:
# MANUAL_DATASET_DIR = '/kaggle/input/agent-eval-scenarios'
MANUAL_DATASET_DIR = ''

input_root = Path('/kaggle/input')
print('Expected AgentEval dataset path =', EXPECTED_DATASET_DIR)
print('Expected path exists =', EXPECTED_DATASET_DIR.exists())

if MANUAL_DATASET_DIR:
    DATASET_DIR = Path(MANUAL_DATASET_DIR)
elif input_root.exists():
    candidates = sorted([p for p in input_root.iterdir() if p.is_dir()])
    if EXPECTED_DATASET_DIR.exists() and EXPECTED_DATASET_DIR not in candidates:
        candidates.insert(0, EXPECTED_DATASET_DIR)
    if not candidates:
        candidates = [input_root]

    print('\nAvailable Kaggle input directories:')
    candidate_diagnostics = {}
    for candidate in candidates:
        diagnostics = dataset_diagnostics(candidate, max_files=8)
        candidate_diagnostics[candidate] = diagnostics
        print('\n---')
        print(candidate)
        print('total_files =', diagnostics['total_files'])
        print('supported_files =', diagnostics['supported_files'])
        for file_path in diagnostics['sample_supported_files']:
            print('  supported:', file_path)

    expected_diagnostics = candidate_diagnostics.get(EXPECTED_DATASET_DIR)
    if expected_diagnostics and expected_diagnostics['supported_files'] > 0:
        DATASET_DIR = EXPECTED_DATASET_DIR
    else:
        DATASET_DIR = max(candidates, key=lambda path: candidate_diagnostics[path]['supported_files'])
else:
    DATASET_DIR = Path('demo-missing-kaggle-input')

print('\nSelected DATASET_DIR =', DATASET_DIR)
selected_diagnostics = dataset_diagnostics(DATASET_DIR, max_files=30)
print(format_dataset_diagnostics(selected_diagnostics))

preview_records = load_agentic_eval_dataset(DATASET_DIR, sample_size=3)
HAS_AGENTIC_EVAL_DATA = bool(preview_records)
print('Preview records loaded =', len(preview_records))
for record in preview_records:
    print('-', record.record_id, 'source=', record.source_path, 'chars=', len(record.combined_text()))

if HAS_AGENTIC_EVAL_DATA:
    print('\nDataset is ready for dry-run, benchmark, and real Gemma cells.')
else:
    print('\nSTOP: No AgentEval records were loaded from DATASET_DIR.')
    sample_files = selected_diagnostics['sample_all_files']
    competition_note_only = (
        selected_diagnostics['supported_files'] == 0
        and sample_files
        and all('/competitions/' in path and path.endswith('/NOTE.md') for path in sample_files)
    )
    if not EXPECTED_DATASET_DIR.exists():
        print('Expected dataset path is missing:', EXPECTED_DATASET_DIR)
        print('In Kaggle, open the notebook right sidebar, click Add data/Input,')
        print('and add dataset mukundakatta/agent-eval-scenarios. Kaggle will mount it')
        print('under /kaggle/input/agent-eval-scenarios/.')
    elif selected_diagnostics['exists'] and selected_diagnostics['total_files'] == 0:
        print('/kaggle/input is empty, so no Kaggle dataset/input is attached to this notebook.')
    elif competition_note_only:
        print('Only the Kaggle competition placeholder NOTE.md is mounted.')
        print('That file is not the AgentEval dataset and cannot be processed.')
    elif selected_diagnostics['supported_files'] == 0:
        print('Files are attached, but none have a supported suffix.')
        print('Set MANUAL_DATASET_DIR to the folder containing CSV/JSON/JSONL/NDJSON/Parquet records,')
        print('or upload the dataset in one of those formats.')
    else:
        print('Supported files were found, but no rows could be parsed. Check the file schema/content.')
    print('Attach the real AgentEval dataset before running dry-run, benchmark, or real Gemma cells.')


## Add and verify your Google API key using Kaggle Secrets

Write the secret in the Kaggle UI, not in this notebook:

1. Open the notebook right sidebar.
2. Click **Add-ons**.
3. Click **Secrets**.
4. Add a secret with label `GEMINI_API_KEY` and value equal to your Google AI Studio / Gemini API key.
5. Turn on notebook access for that secret.

The code cell below is where you verify the secret and write `/kaggle/working/configs/secrets.toml`, which is what the pipeline reads. It prints only `True`/`False`, never the key itself.


In [ ]:
config_dir = Path('/kaggle/working/configs') if Path('/kaggle/working').exists() else REPO_ROOT / 'configs'
config_dir.mkdir(parents=True, exist_ok=True)

# This label must exactly match the label you added in Kaggle Add-ons -> Secrets.
secret_label = 'GEMINI_API_KEY'
api_key = ''
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret(secret_label)
except Exception as exc:
    print(f'Could not load Kaggle Secret {secret_label!r}: {exc}')

print('API key loaded from Kaggle Secrets =', bool(api_key))

app_toml = f'''
[gemma]
model = "gemma-4-26b-a4b-it"
temperature = 0.2
top_p = 0.9
max_output_tokens = 2048

[data]
input_dir = "{DATASET_DIR}"
sample_size = 0

[pruning]
alpha = 1.5
beta = 1.0
gamma = 0.5
delta = 10.0
prune_ratio = 0.45
cache_threshold = 2
priority_boost = 1.5

[output]
path = "/kaggle/working/results.jsonl"
'''.strip() + "\n"

secrets_toml = f'''
[gemma]
api_key = "{api_key}"
'''.strip() + "\n"

(config_dir / 'app.toml').write_text(app_toml, encoding='utf-8')
(config_dir / 'secrets.toml').write_text(secrets_toml, encoding='utf-8')

print('Wrote', config_dir / 'app.toml')
print('Wrote', config_dir / 'secrets.toml')
print('API key present =', bool(api_key))


## Patch Gemma JSON parsing in notebook runtime

The repository already includes robust JSON parsing. This cell also patches the currently imported runtime class, which helps if Kaggle imported `acpa_gemma` before the repo update. It forces JSON-only instructions and extracts a JSON object from noisy model output when possible.


In [ ]:
import json
import re

from acpa_gemma import gemma_client


def robust_generate_json(self, prompt, system_instruction=None):
    """
    Wrap Gemma generate + parsing so that:
    - JSON-only style is forced through system_instruction
    - A JSON object is extracted from noisy output when possible
    """
    json_system_instruction = (
        'You are a JSON-only assistant. '
        'You MUST respond with a single valid JSON object. '
        'Do not include explanations, markdown, or any text before or after the JSON. '
        'If you want to explain something, put it inside JSON string fields. '
    )

    if system_instruction:
        system_instruction = json_system_instruction + ' ' + str(system_instruction)
    else:
        system_instruction = json_system_instruction

    text = self.generate(prompt=prompt, system_instruction=system_instruction)

    try:
        return gemma_client.parse_json_object(text)
    except Exception:
        print('DEBUG RAW GEMMA OUTPUT:\n', text[:1000])
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if not match:
            snippet = text[:500].replace('\n', ' ')
            raise ValueError(f'Gemma output must be a JSON object, got: {snippet}')

        json_str = match.group(0)
        try:
            payload = json.loads(json_str)
        except Exception as exc:
            snippet = json_str[:500].replace('\n', ' ')
            raise ValueError(
                'Failed to parse JSON from Gemma output. '
                f'Extracted: {snippet}. Error: {exc}'
            ) from exc

        if not isinstance(payload, dict):
            raise ValueError(f'Gemma output must be a JSON object, got type={type(payload)}')

        return payload


gemma_client.GemmaClient.generate_json = robust_generate_json
print('Patched GemmaClient.generate_json to robust JSON parser')


## Dry-run pipeline test, no API key required

This verifies dataset loading, context construction, ACPA pruning, and JSONL output using attached AgentEval records. It stops if the AgentEval dataset is not attached.


In [ ]:
from acpa_gemma.cli import main as pipeline_main

if not HAS_AGENTIC_EVAL_DATA:
    raise RuntimeError(
        'Attach mukundakatta/agent-eval-scenarios in Kaggle Add data/Input, '
        'then rerun the dataset locator cell before dry-run.'
    )

pipeline_main([
    '--config', str(config_dir / 'app.toml'),
    '--secrets', str(config_dir / 'secrets.toml'),
    '--input', str(DATASET_DIR),
    '--output', '/kaggle/working/dry_run.jsonl',
    '--sample-size', '3',
    '--dry-run',
])

print(Path('/kaggle/working/dry_run.jsonl').read_text(encoding='utf-8')[:2000])


## Benchmark ACPA vs baseline algorithms, no API key required

Creates per-record CSV, aggregate CSV, and a Markdown report using attached AgentEval records only.


In [ ]:
from acpa_gemma.benchmark import main as benchmark_main

if not HAS_AGENTIC_EVAL_DATA:
    raise RuntimeError(
        'Attach mukundakatta/agent-eval-scenarios in Kaggle Add data/Input, '
        'then rerun the dataset locator cell before benchmark.'
    )

benchmark_main([
    '--input', str(DATASET_DIR),
    '--sample-size', '100',
    '--details-output', '/kaggle/working/benchmark_details.csv',
    '--summary-output', '/kaggle/working/benchmark_summary.csv',
    '--report-output', '/kaggle/working/benchmark_report.md',
])

print(Path('/kaggle/working/benchmark_report.md').read_text(encoding='utf-8'))


## Real Gemma 4 run

Run this after `API key present = True`. Start with a small sample, then remove `--sample-size` for the full run.


In [ ]:
from acpa_gemma.cli import main as pipeline_main

if not api_key:
    raise RuntimeError('Add GEMINI_API_KEY/GEMMA_API_KEY in Kaggle Secrets or paste api_key in the config cell first.')
if not HAS_AGENTIC_EVAL_DATA:
    raise RuntimeError(
        'No Agentic Eval records are attached. Add the dataset in Kaggle Add data, '
        'or set DATASET_DIR to a directory containing CSV/JSON/JSONL/NDJSON/Parquet files, '
        'then rerun the dataset locator cell before calling the real Gemma API.'
    )

pipeline_main([
    '--config', str(config_dir / 'app.toml'),
    '--secrets', str(config_dir / 'secrets.toml'),
    '--input', str(DATASET_DIR),
    '--output', '/kaggle/working/results_sample.jsonl',
    '--sample-size', '10',
])

print(Path('/kaggle/working/results_sample.jsonl').read_text(encoding='utf-8')[:3000])


## Full results run

Uncomment this cell when the sample run looks good.


In [ ]:
# pipeline_main([
#     '--config', str(config_dir / 'app.toml'),
#     '--secrets', str(config_dir / 'secrets.toml'),
#     '--input', str(DATASET_DIR),
#     '--output', '/kaggle/working/results.jsonl',
# ])


## Free/offline CUAD usage-driven pruning benchmark

This section does **not** call Gemma and does **not** require an API key. It uses CUAD `data.zip` from The Atticus Project, runs the usage-driven context pruning benchmark against five SOTA-style baselines, and writes journal-style CSV, Markdown, and SVG plot artifacts.

If you already attached CUAD as a Kaggle input, the code will use the mounted `data.zip`. Otherwise it downloads CUAD from GitHub to `/kaggle/working/cuad_data.zip`.

Set `CUAD_MAX_CONTRACTS = 0` for a full CUAD run. Start with `50` or lower while validating runtime.


In [ ]:
from pathlib import Path
import urllib.request

from acpa_gemma.cuad import CUAD_DATA_URL, main as cuad_main

CUAD_MAX_CONTRACTS = 50  # Set to 0 for full CUAD after the smoke run looks good.
CUAD_PRUNE_RATIOS = '0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8'
CUAD_POLICIES = ','.join([
    'usage_driven',
    'hybrid_usage_bm25',
    'bm25_query_relevance',
    'mmr_diverse_relevance',
    'rrf_bm25_textrank',
    'dpp_diverse_relevance',
    'late_interaction_maxsim',
])

input_root = Path('/kaggle/input')
cuad_candidates = []
if input_root.exists():
    cuad_candidates = sorted(
        path for path in input_root.rglob('data.zip')
        if 'cuad' in str(path).lower() or 'atticus' in str(path).lower()
    )
    if not cuad_candidates:
        cuad_candidates = sorted(input_root.rglob('data.zip'))

if cuad_candidates:
    CUAD_INPUT = cuad_candidates[0]
    print('Using mounted CUAD zip:', CUAD_INPUT)
else:
    CUAD_INPUT = Path('/kaggle/working/cuad_data.zip') if Path('/kaggle/working').exists() else Path('cuad_data.zip')
    if not CUAD_INPUT.exists():
        print('Downloading CUAD data.zip from:', CUAD_DATA_URL)
        urllib.request.urlretrieve(CUAD_DATA_URL, CUAD_INPUT)
    print('Using downloaded CUAD zip:', CUAD_INPUT)

cuad_args = [
    '--input', str(CUAD_INPUT),
    '--json-member', 'CUADv1.json',
    '--train-fraction', '0.6',
    '--prune-ratios', CUAD_PRUNE_RATIOS,
    '--policies', CUAD_POLICIES,
    '--summary-output', '/kaggle/working/cuad_summary.csv',
    '--details-output', '/kaggle/working/cuad_details.csv',
    '--report-output', '/kaggle/working/cuad_report.md',
    '--plots-output-dir', '/kaggle/working/cuad_plots',
]
if CUAD_MAX_CONTRACTS:
    cuad_args.extend(['--max-contracts', str(CUAD_MAX_CONTRACTS)])

cuad_main(cuad_args)

print(Path('/kaggle/working/cuad_report.md').read_text(encoding='utf-8')[:5000])
print('\nCUAD artifacts:')
for output_path in [
    '/kaggle/working/cuad_summary.csv',
    '/kaggle/working/cuad_details.csv',
    '/kaggle/working/cuad_report.md',
    '/kaggle/working/cuad_plots/citation_accuracy_vs_context_removed.svg',
    '/kaggle/working/cuad_plots/answer_quality_vs_context_removed.svg',
    '/kaggle/working/cuad_plots/improvement_vs_context_removed.svg',
    '/kaggle/working/cuad_plots/max_safe_context_removed_by_policy.svg',
]:
    path = Path(output_path)
    print(output_path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else 0)


## Output files

Attach these files to your research report, hackathon demo, or publication appendix as needed:

AgentEval / Gemma outputs:

- `/kaggle/working/results.jsonl`
- `/kaggle/working/results_sample.jsonl`
- `/kaggle/working/benchmark_report.md`
- `/kaggle/working/benchmark_summary.csv`
- `/kaggle/working/benchmark_details.csv`

Free/offline CUAD benchmark outputs:

- `/kaggle/working/cuad_summary.csv`
- `/kaggle/working/cuad_details.csv`
- `/kaggle/working/cuad_report.md`
- `/kaggle/working/cuad_plots/citation_accuracy_vs_context_removed.svg`
- `/kaggle/working/cuad_plots/answer_quality_vs_context_removed.svg`
- `/kaggle/working/cuad_plots/improvement_vs_context_removed.svg`
- `/kaggle/working/cuad_plots/max_safe_context_removed_by_policy.svg`
